# 10年定着予測 - テキスト特徴量を事前学習済み文埋め込みモデルに置き換え

**背景**: これまで入社時メモ・上司/同僚フィードバックには文字n-gram TF-IDF + TruncatedSVDを使ってきたが、
分割間でスコアがブレやすい（stdがbaselineの2〜3倍）という弱点があった（`17_`）。今回、日本語の
事前学習済み文埋め込みモデルに置き換えることで、次元を増やさず意味的な情報を捉え、この弱点を
改善できるか検証する。

## モデル選定の経緯

日本語特化のSentence-BERT（`sonoisa/sentence-bert-base-ja-mean-tokens-v2`）を最初に検討したが、
**MeCab（`fugashi`）による形態素解析が必要**で、Colab環境への追加インストールが不安定になりやすいことが
分かった（実際にローカルで検証時にエラーが発生した）。

代わりに、**分かち書き不要な多言語モデル `intfloat/multilingual-e5-small`**（SentencePieceベースの
トークナイザで日本語もそのまま扱える、384次元）を採用する。E5系モデルの慣例に従い、
文書には`"passage: "`という接頭辞を付けてエンコードする。

384次元のままでは特徴量が多すぎるため、PCA（`svd_solver='full'`、Trainのみでfit）で15次元に圧縮し、
TF-IDF+SVDの次元数（15次元）と揃えて公平に比較する。

## 実施すること

`D_expanded`（`18_`で確立した最良のD構成）を固定した上で、テキスト特徴量の扱いを3パターンで比較する
（80/20・75/25の2 splitで評価、CatBoostのみ使用 — `18_`でCatBoostが大差で最良と判明済みのため）。

| 設定名 | テキスト特徴量 |
|---|---|
| `no_text` | なし（文字数のみ） |
| `tfidf` | 文字n-gram TF-IDF + SVD（15次元、`17_`と同じ） |
| `embedding` | multilingual-e5-small + PCA（15次元、新規） |

## 実行環境
Google Colab（GPU: T4）を想定。`sentence-transformers`の追加インストールが必要。

In [1]:
!pip install -q catboost optuna sentence-transformers

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Aug  9 15:13:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             32W /   70W |     371MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import datetime
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
import torch
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
SCRIPT_NAME = "19_text_embeddings"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# チェックポイント（日付非依存の固定パス。セッションをまたいで再開できるようにする）
CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")

[2026-08-09 15:13:29] [INFO] === [19_text_embeddings] 実験開始 ===


INFO:19_text_embeddings:=== [19_text_embeddings] 実験開始 ===


[2026-08-09 15:13:29] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


INFO:19_text_embeddings:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260809


[2026-08-09 15:13:29] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/19_text_embeddings_checkpoint.csv


INFO:19_text_embeddings:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/19_text_embeddings_checkpoint.csv


[2026-08-09 15:13:29] [INFO] チェックポイントは未作成（新規実行）


INFO:19_text_embeddings:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-09 15:13:30] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:19_text_embeddings:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-09 15:13:30] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:19_text_embeddings:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-09 15:13:30] [INFO] 定着率: 0.5647


INFO:19_text_embeddings:定着率: 0.5647


[2026-08-09 15:13:30] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:19_text_embeddings:Train IDs: 2761, Test IDs: 2502


## 1. 基本特徴量関数の定義（split非依存）

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)


def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ split非依存の基本特徴量関数定義完了")

✅ split非依存の基本特徴量関数定義完了


In [8]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

# 18_で確立した最良のDブロック（月次全16指標の四半期/加速度特徴量）を固定で使用
D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")

logger.info("split非依存の基本特徴量生成完了")

[2026-08-09 15:13:31] [INFO] ------------------------------------------------------------


INFO:19_text_embeddings:------------------------------------------------------------


[2026-08-09 15:13:31] [INFO] split非依存の基本特徴量を生成中...


INFO:19_text_embeddings:split非依存の基本特徴量を生成中...


[2026-08-09 15:13:31] [INFO] ------------------------------------------------------------


INFO:19_text_embeddings:------------------------------------------------------------


[2026-08-09 15:23:35] [INFO] split非依存の基本特徴量生成完了


INFO:19_text_embeddings:split非依存の基本特徴量生成完了


## 2. テキスト特徴量A: TF-IDF + SVD（`17_`と同一、比較のベースライン）

In [9]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    """文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）"""
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)
logger.info("テキストTF-IDF+SVD特徴量生成完了")

[2026-08-09 15:23:36] [INFO] テキストTF-IDF+SVD特徴量を生成中...


INFO:19_text_embeddings:テキストTF-IDF+SVD特徴量を生成中...


[2026-08-09 15:23:44] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:19_text_embeddings:テキストTF-IDF+SVD特徴量生成完了


## 3. テキスト特徴量B: 事前学習済み文埋め込み（新規）

`intfloat/multilingual-e5-small`（384次元、分かち書き不要）でテキストをベクトル化し、
PCA（`svd_solver='full'`、Trainのみでfit）で15次元に圧縮する。E5系モデルの慣例に従い
`"passage: "`という接頭辞を付ける。GPUが使えれば自動的に使用される。

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"文埋め込みモデルのデバイス: {device}")

embed_model = SentenceTransformer("intfloat/multilingual-e5-small", device=device)

def create_embedding_pca_features(train_persona, test_persona, col, model, n_components=15, seed=42):
    """事前学習済み文埋め込み(e5) + PCAでテキスト特徴量を生成（Trainのみでfit）"""
    train_text = ("passage: " + train_persona[col].fillna("").astype(str)).tolist()
    test_text = ("passage: " + test_persona[col].fillna("").astype(str)).tolist()

    train_emb = model.encode(train_text, batch_size=64, show_progress_bar=False)
    test_emb = model.encode(test_text, batch_size=64, show_progress_bar=False)

    pca = PCA(n_components=n_components, random_state=seed, svd_solver="full")
    train_pca = pca.fit_transform(train_emb)
    test_pca = pca.transform(test_emb)

    col_names = [f"{col}_emb_pca_{i}" for i in range(n_components)]
    train_out = pd.DataFrame(train_pca, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_pca, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, pca.explained_variance_ratio_.sum()

logger.info("事前学習済み文埋め込み特徴量を生成中...")
embed_train_list, embed_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_embedding_pca_features(train_persona, test_persona, col, embed_model, n_components=15, seed=SEED)
    logger.info(f"{col}: PCA累積寄与率={explained_var:.3f}")
    embed_train_list.append(tr)
    embed_test_list.append(te)
logger.info("事前学習済み文埋め込み特徴量生成完了")

[2026-08-09 15:23:44] [INFO] 文埋め込みモデルのデバイス: cuda


INFO:19_text_embeddings:文埋め込みモデルのデバイス: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

INFO:19_text_embeddings:上司からのフィードバック: PCA累積寄与率=0.398


[2026-08-09 15:25:03] [INFO] 同僚からのフィードバック: PCA累積寄与率=0.524


INFO:19_text_embeddings:同僚からのフィードバック: PCA累積寄与率=0.524


[2026-08-09 15:25:03] [INFO] 事前学習済み文埋め込み特徴量生成完了


INFO:19_text_embeddings:事前学習済み文埋め込み特徴量生成完了


## 4. Persona単位の基本特徴量（split非依存）

In [11]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")

[2026-08-09 15:25:03] [INFO] Persona単位の基本特徴量を生成中...


INFO:19_text_embeddings:Persona単位の基本特徴量を生成中...


[2026-08-09 15:25:03] [INFO] Persona単位の基本特徴量処理完了


INFO:19_text_embeddings:Persona単位の基本特徴量処理完了


## 5. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`text_variant`パラメータで `None`/`"tfidf"`/`"embedding"` を切り替えられるようにする。
Dブロックは`18_`で確立した`D_expanded`を常に使用する。

In [12]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）"""
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, text_variant=None):
    """指定した分割比率で特徴量を組み立てる。text_variant: None/"tfidf"/"embedding" """
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")

    if text_variant == "tfidf":
        for trdf in tfidf_train_list:
            tf = tf.merge(trdf, on=ID_COL, how="left")
        for tedf in tfidf_test_list:
            ttf = ttf.merge(tedf, on=ID_COL, how="left")
    elif text_variant == "embedding":
        for trdf in embed_train_list:
            tf = tf.merge(trdf, on=ID_COL, how="left")
        for tedf in embed_test_list:
            ttf = ttf.merge(tedf, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df in [tf, ttf]:
        for m in job_dev_metrics:
            df[f"{m}_job_deviation"] = df[m] - df["初期職種"].map(job_means[m])
        df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
        df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
        df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")

✅ 部署Target Encoding・prepare_split関数定義完了


## 6. チェックポイント機能（`18_`と同一）

計算済みの設定はスキップし、未計算の設定のみ実行する。

In [13]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=["config", "n_features", "val_score", "submission_path"])

def save_checkpoint_row(result):
    df = pd.DataFrame([result])
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)

def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label]
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_score={row['val_score']:.6f}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")

✅ チェックポイント関数定義完了


## 7. CatBoost実行関数

`18_`でCatBoostがLightGBM/XGBoostを大差で上回ることが判明したため、本ノートブックはCatBoostのみ使用する。

In [14]:
def run_catboost_config(ag_train_data, ag_tuning_data, test_features, config_label, n_trials=25):
    feature_cols = [c for c in ag_train_data.columns if c not in ["入社日", TARGET_COL]]
    cat_cols = [c for c in feature_cols if ag_train_data[c].dtype == "object"]

    X_tr = ag_train_data[feature_cols].fillna(-999)
    y_tr = ag_train_data[TARGET_COL]
    X_va = ag_tuning_data[feature_cols].fillna(-999)
    y_va = ag_tuning_data[TARGET_COL]

    def objective(trial):
        params = {
            "depth": trial.suggest_int("depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "iterations": 1000, "random_seed": SEED, "verbose": False,
            "cat_features": cat_cols, "early_stopping_rounds": 50, "task_type": "GPU",
        }
        model = cb.CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        return log_loss(y_va, model.predict_proba(X_va)[:, 1])

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params

    final_model = cb.CatBoostClassifier(
        **best_params, iterations=3000, random_seed=SEED, verbose=False,
        cat_features=cat_cols, early_stopping_rounds=100, task_type="GPU",
    )
    final_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    val_preds = final_model.predict_proba(X_va)[:, 1]
    test_preds = final_model.predict_proba(test_features[feature_cols].fillna(-999))[:, 1]

    val_score = log_loss(y_va, val_preds)
    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    sub = pd.DataFrame({ID_COL: test_features.index, TARGET_COL: test_preds})
    sub.to_csv(sub_path, index=False, header=False)

    logger.info(f"[{config_label}] n_features={len(feature_cols)}, val_score={val_score:.6f}")
    return {"config": config_label, "n_features": len(feature_cols), "val_score": val_score, "submission_path": str(sub_path)}

print("✅ run_catboost_config関数定義完了")

✅ run_catboost_config関数定義完了


## 8. ステップA: テキスト特徴量比較（なし / TF-IDF / 文埋め込み）× 2 split

`D_expanded`固定の上で、テキスト特徴量の扱いを3パターン、80/20・75/25の両方で評価する。

In [15]:
SPLIT_RATIOS = {"split_80_20": 0.8, "split_75_25": 0.75}
TEXT_VARIANTS = {"no_text": None, "tfidf": "tfidf", "embedding": "embedding"}

stepA_results = []
for split_name, ratio in SPLIT_RATIOS.items():
    for variant_name, text_variant in TEXT_VARIANTS.items():
        config_label = f"stepA_{split_name}_{variant_name}"
        def _run(ratio=ratio, text_variant=text_variant, config_label=config_label):
            logger.info(f"=== {config_label} ===")
            ag_train_data, ag_tuning_data, test_features_full = prepare_split(ratio, text_variant=text_variant)
            return run_catboost_config(ag_train_data, ag_tuning_data, test_features_full, config_label, n_trials=25)
        result = run_or_resume(config_label, _run)
        result["split"] = split_name
        result["variant"] = variant_name
        stepA_results.append(result)

stepA_df = pd.DataFrame(stepA_results)
stepA_pivot = stepA_df.pivot(index="variant", columns="split", values="val_score")
stepA_pivot["mean"] = stepA_pivot.mean(axis=1)
stepA_pivot["std"] = stepA_pivot[["split_80_20", "split_75_25"]].std(axis=1)
stepA_pivot = stepA_pivot.sort_values("mean")

logger.info("=" * 60)
logger.info("ステップA結果（テキスト特徴量比較）")
logger.info("=" * 60)
logger.info("\n" + stepA_pivot.to_string())
print("\n■ ステップA結果:")
print(stepA_pivot.to_string())
print(f"\n(参考) 18_ CatBoost+D_expanded（テキストはTF-IDF, split_80_20）: val 0.537033, Public 0.550352（現時点の最良）")

best_variant_name = stepA_pivot["mean"].idxmin()
logger.info(f"選ばれたテキスト特徴量: {best_variant_name}")
print(f"\n■ 選ばれたテキスト特徴量: {best_variant_name}")

[2026-08-09 15:25:04] [INFO] === stepA_split_80_20_no_text ===


INFO:19_text_embeddings:=== stepA_split_80_20_no_text ===


[2026-08-09 15:43:15] [INFO] [stepA_split_80_20_no_text] n_features=394, val_score=0.541300


INFO:19_text_embeddings:[stepA_split_80_20_no_text] n_features=394, val_score=0.541300


[2026-08-09 15:43:15] [INFO] === stepA_split_80_20_tfidf ===


...

[2026-08-09 17:22:46] [INFO] 
split      split_75_25  split_80_20      mean       std
variant                                                
tfidf         0.551960     0.536748  0.544354  0.010757
embedding     0.551507     0.541603  0.546555  0.007003
no_text       0.556170     0.541300  0.548735  0.010515


INFO:19_text_embeddings:
split      split_75_25  split_80_20      mean       std
variant                                                
tfidf         0.551960     0.536748  0.544354  0.010757
embedding     0.551507     0.541603  0.546555  0.007003
no_text       0.556170     0.541300  0.548735  0.010515



■ ステップA結果:
split      split_75_25  split_80_20      mean       std
variant                                                
tfidf         0.551960     0.536748  0.544354  0.010757
embedding     0.551507     0.541603  0.546555  0.007003
no_text       0.556170     0.541300  0.548735  0.010515

(参考) 18_ CatBoost+D_expanded（テキストはTF-IDF, split_80_20）: val 0.537033, Public 0.550352（現時点の最良）
[2026-08-09 17:22:46] [INFO] 選ばれたテキスト特徴量: tfidf


INFO:19_text_embeddings:選ばれたテキスト特徴量: tfidf



■ 選ばれたテキスト特徴量: tfidf


## 9. まとめ・次のアクション

1. ステップAの表で、`no_text` / `tfidf` / `embedding` のmean（2 split平均）とstd（分割間のブレ）を比較する
2. `embedding`が`tfidf`よりmeanが良い、またはstdが小さければ、テキスト表現の置き換えが有効だったと言える
3. 最良の設定（`stepA_split_80_20_{best_variant_name}`のファイル）をKaggleに提出し、Publicスコアを確認する
4. 結果が出たら`submit_result_report.md`に追記する